In [ ]:
%pip install shap

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" 

import shap
import torch
from transformers import pipeline

# 1. load model
model_name = "roberta-large-mnli"
classifier = pipeline("text-classification", model=model_name, top_k=None)

# 2. Definition of an interpreter
explainer = shap.Explainer(classifier)

# 3. prepare the example
test_texts = [
    "This is a bread knife. </s> </s> The knife is used for the purpose of bread.",
    "This is a steel table. </s> </s> The table contains or holds the steel."
]

# 4. SHAP analysis
shap_values = explainer(test_texts)

# 5. Visualization
shap.plots.text(shap_values[0, :, "ENTAILMENT"])

In [ ]:
# Sampling samples with prediction errors

import os
import json
import shap
from transformers import pipeline

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def analyze_errors_with_shap(results_file="nli_results_full.jsonl", top_n=3):
    # 1. load the prediction errors
    error_samples = []
    
    with open(results_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # we expect ENTAILMENT
            # if the prediction is not ENTAILMENT，then is error
            if data.get('predicted_nli') != 'ENTAILMENT':
                error_samples.append(data)

    if not error_samples:
        print("no prediction error")
        return

    # Sort by confidence in descending order to identify the samples where the model is “most confident but completely wrong”
    error_samples.sort(key=lambda x: x.get('conf_score', 0), reverse=True)
    target_samples = error_samples[:top_n]
    

    # 2. Prepare the SHAP interpreter
    
    classifier = pipeline("text-classification", model="roberta-large-mnli", device=-1, top_k=None)
    explainer = shap.Explainer(classifier)

    # 3. Extract text format
    # Must be formatted in a way the model can recognize: Premise </s> </s> Hypothesis
    texts_to_analyze = []
    for data in target_samples:
        text = f"{data['premise']} </s> </s> {data['hypothesis']}"
        texts_to_analyze.append(text)
        print(f"Objective [{data['original_label']}]: {data['compound']} (predicted {data['predicted_nli']})")

    # 4. SHAP analysis
    shap_values = explainer(texts_to_analyze)

    # 5. Generate and save the visualization as HTML
    html_content = shap.plots.text(shap_values, display=False)
    
    output_html = "shap_error_analysis.html"
    with open(output_html, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'><title>NLI SHAP Error Analysis</title></head><body style='padding: 20px;'>")
        f.write("<h2>NLI error SHAP report</h2>")
        f.write("<p>Words highlighted in red are those that contribute to this result, while those highlighted in blue are those that detract from it.</p>")
        f.write(html_content)
        f.write("</body></html>")
        
    print(f" SHAP analysis finished")
    print(f"open the html report：{os.path.abspath(output_html)}")

if __name__ == "__main__":
    analyze_errors_with_shap("nli_results_full.jsonl", top_n=3)